# FX Carry Trade with a Momentum Filter
### A systematic macro strategy on G10 currencies: theory, backtest and live signal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabsmm18/fx-carry-momentum/blob/main/notebooks/carry_momentum_fx.ipynb)

**Author:** Mattia Mariani — MSc Applied Economics and Finance, Copenhagen Business School

**Research question.** The FX carry trade has historically earned a positive average return, but it suffers sudden *crashes* (2008, 2015, August 2024). Does a simple momentum filter, which closes carry positions when the trend turns against them, reduce these crashes? And at what cost?

**Outline**
1. Theory: why carry *should not* work (UIP) and why it does
2. Data: OECD 3-month rates and Fed H.10 FX rates, all free from FRED
3. Signal construction: carry, momentum and the filter
4. Backtest with transaction costs
5. Analysis: crisis episodes, robustness, sub-samples
6. Live signal: positions for next month
7. Limitations

> ⚠️ Educational and research project. Not investment advice. A backtest is no guarantee of future returns.

**To run:** `Runtime → Run all` (about 30 seconds, no API key needed).

---
## 1. Theory

### 1.1 Uncovered Interest Parity (UIP)
Let $S_t$ be the exchange rate in **USD per 1 unit of foreign currency** (a rise in $S$ = the foreign currency appreciates), $i_t$ the US interest rate and $i^*_t$ the foreign rate.

Investing 1 USD abroad for one period yields, in USD, approximately:
$$ r^{\text{carry}}_{t+1} \approx \underbrace{(i^*_t - i_t)}_{\text{interest differential (known today)}} + \underbrace{\Delta \ln S_{t+1}}_{\text{FX change (unknown)}} $$

**UIP** states that, with risk-neutral investors, the expected value of this return is zero: the high-yielding currency should **depreciate** on average by exactly the extra interest it pays:
$$ E_t[\Delta \ln S_{t+1}] = -(i^*_t - i_t) $$

### 1.2 The forward premium puzzle (Fama, 1984)
The data say the opposite. Estimating $\Delta \ln S_{t+1} = \alpha + \beta (i_t - i^*_t) + \varepsilon$, UIP implies $\beta = 1$, but typical estimates are **$\beta < 1$, often negative**. On average, high-yielding currencies **do not depreciate enough**, and sometimes even appreciate. Therefore:

> **Buying high-yield currencies funded with low-yield currencies has a positive average return.** This is the carry trade.

### 1.3 Why does carry pay? A crash-risk premium
If it were a free lunch, it would be arbitraged away. The dominant explanation in the literature is a **risk premium**:
- **Crash risk** (Brunnermeier, Nagel & Pedersen, 2008): carry returns are **negatively skewed**. Small, steady gains and rare, violent losses when crowded positions are unwound all at once ("picking up nickels in front of a steamroller").
- **Global volatility risk** (Menkhoff, Sarno, Schmeling & Schrimpf, 2012, *JF*): carry loses exactly when global FX volatility spikes, i.e. in bad times for investors, so they demand compensation.
- **Common risk factor** (Lustig, Roussanov & Verdelhan, 2011, *RFS*): a currency "HML" factor (long high-yielders, short low-yielders) explains much of the cross-section of FX returns.

### 1.4 Currency momentum
Menkhoff, Sarno, Schmeling & Schrimpf (2012, *JFE*) document that currencies that have recently appreciated tend to keep doing so for a few months. Explanations range from under-reaction to information to gradual central-bank behaviour.

### 1.5 The idea of this strategy
The two signals play different roles:
- **Carry** tells us **WHAT** to hold: the currencies paying the most.
- **Momentum** tells us **WHEN NOT** to hold them: if a high-yielder is already depreciating (negative momentum), an unwind is probably under way and the interest differential will not compensate for the FX loss.

Rule:
| Carry position | Last 3-month momentum | Action |
|---|---|---|
| Long (high yield) | > 0 | keep long |
| Long (high yield) | < 0 | **flat** |
| Short (low yield) | < 0 | keep short |
| Short (low yield) | > 0 | **flat** |

Hypothesis to test: the filter **reduces drawdowns and negative skewness**, at the cost of a slightly lower return in calm periods.

---
## 2. Setup and data

We only use **FRED** (Federal Reserve Bank of St. Louis), free and without an API key:
- **FX rates**: Fed daily noon rates (H.10 release), sampled at month-end.
- **Interest rates**: OECD 3-month interbank rates (Main Economic Indicators), monthly.

Universe: the 9 G10 currencies vs USD (EUR, GBP, JPY, CHF, AUD, NZD, CAD, NOK, SEK).

In [ ]:
import io
import json
import os
import time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (11, 5), 'axes.grid': True, 'grid.alpha': 0.3})
pd.set_option('display.float_format', '{:.3f}'.format)

### 2.1 Series definitions and download functions
An important detail: FRED quotes some currencies as *USD per foreign unit* (EUR, GBP, AUD, NZD) and others as *foreign units per USD* (JPY, CHF, CAD, NOK, SEK). To use one convention, the latter are **inverted**, so that for every currency "up = the foreign currency appreciates".

OECD rates are published with a 1–3 month lag. For the most recent months we carry the last available value forward (*forward fill*) for at most 4 months. This is reasonable because the 3-month rate is highly persistent, but it must be stated.

In [ ]:
# FRED FX series: (series id, True if quoted as foreign per USD -> must invert)
FX_SERIES = {
    "EUR": ("DEXUSEU", False),  # USD per EUR
    "GBP": ("DEXUSUK", False),  # USD per GBP
    "AUD": ("DEXUSAL", False),  # USD per AUD
    "NZD": ("DEXUSNZ", False),  # USD per NZD
    "JPY": ("DEXJPUS", True),   # JPY per USD -> invert
    "CHF": ("DEXSZUS", True),   # CHF per USD -> invert
    "CAD": ("DEXCAUS", True),   # CAD per USD -> invert
    "NOK": ("DEXNOUS", True),   # NOK per USD -> invert
    "SEK": ("DEXSDUS", True),   # SEK per USD -> invert
}

# OECD 3-month interbank rates (monthly averages, % p.a.) via FRED
RATE_SERIES = {
    "USD": "IR3TIB01USM156N",
    "EUR": "IR3TIB01EZM156N",
    "GBP": "IR3TIB01GBM156N",
    "JPY": "IR3TIB01JPM156N",
    "CHF": "IR3TIB01CHM156N",
    "AUD": "IR3TIB01AUM156N",
    "NZD": "IR3TIB01NZM156N",
    "CAD": "IR3TIB01CAM156N",
    "NOK": "IR3TIB01NOM156N",
    "SEK": "IR3TIB01SEM156N",
}

FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"
FRED_API_URL = ("https://api.stlouisfed.org/fred/series/observations"
                "?series_id={}&api_key={}&file_type=json")

# Free key from https://fredaccount.stlouisfed.org/apikeys
# Set it as an environment variable FRED_API_KEY (or assign it here at runtime).
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")


def _http_get(url: str, timeout: int = 60) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8")


def fetch_fred(series_id: str, retries: int = 3) -> pd.Series:
    """
    Download one FRED series as a pandas Series.

    * If FRED_API_KEY is set -> official FRED API (reliable, recommended).
    * Otherwise -> the keyless CSV export, which is often slow or blocked
      from cloud servers such as Google Colab or GitHub Actions.
    Retries with increasing waits if the server does not answer.
    """
    last_err = None
    for attempt in range(retries):
        try:
            if FRED_API_KEY:
                obs = json.loads(_http_get(FRED_API_URL.format(series_id, FRED_API_KEY)))["observations"]
                s = pd.Series({pd.Timestamp(o["date"]): o["value"] for o in obs})
            else:
                raw = _http_get(FRED_CSV_URL.format(series_id))
                df = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True)
                s = df.iloc[:, 0]
            s = pd.to_numeric(s.replace(".", np.nan), errors="coerce").dropna().sort_index()
            s.name = series_id
            return s
        except Exception as e:  # timeout, HTTP error, bad key...
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"{last_err}")


def to_monthly_last(s: pd.Series) -> pd.Series:
    """Daily series -> month-end value, indexed by monthly Period."""
    return s.groupby(s.index.to_period("M")).last()


def to_monthly_period(s: pd.Series) -> pd.Series:
    """Monthly FRED series (dated 1st of month) -> monthly Period index."""
    out = s.copy()
    out.index = out.index.to_period("M")
    return out.groupby(level=0).last()


def load_data(fetch=fetch_fred, start: str = "2000-01", max_ffill_months: int = 4,
              drop_incomplete_month: bool = True, verbose: bool = True):
    """
    Returns
    -------
    spot  : DataFrame (monthly, USD per 1 foreign unit), columns = currencies
    rates : DataFrame (monthly, % p.a.), columns = currencies + 'USD'

    `max_ffill_months`: OECD rates are published with a lag of 1-3 months;
    we carry the last value forward for at most this many months
    (the 3M rate is a very persistent variable, so this is a mild assumption).
    `drop_incomplete_month`: drop the current calendar month (its "month-end"
    price would just be today's price, not a real month-end).
    """
    spot, rates = {}, {}
    for ccy, (sid, invert) in FX_SERIES.items():
        try:
            s = fetch(sid)
            s = 1.0 / s if invert else s
            spot[ccy] = to_monthly_last(s)
        except Exception as e:  # keep going if one series fails
            print(f"[warning] FX {ccy} ({sid}) not downloaded: {e}")
    for ccy, sid in RATE_SERIES.items():
        try:
            rates[ccy] = to_monthly_period(fetch(sid))
        except Exception as e:
            print(f"[warning] rate {ccy} ({sid}) not downloaded: {e}")

    if "USD" not in rates or not spot:
        raise RuntimeError(
            "Download failed (USD rate or all FX series missing). If you see timeouts, "
            "set a free FRED API key: https://fredaccount.stlouisfed.org/apikeys")

    spot = pd.DataFrame(spot).sort_index()
    rates = pd.DataFrame(rates).sort_index()

    if verbose:
        print("Last available observation per series:")
        last = pd.concat(
            [spot.apply(lambda c: c.last_valid_index()).rename("FX"),
             rates.apply(lambda c: c.last_valid_index()).rename("3M rate")],
            axis=1,
        )
        print(last.to_string())

    if drop_incomplete_month:
        this_month = pd.Timestamp.today().to_period("M")
        spot = spot.loc[spot.index < this_month]

    idx = spot.index
    rates =rates.reindex(idx.union(rates.index)).ffill(limit=max_ffill_months).reindex(idx)
    start_p = pd.Period(start, "M")
    return spot.loc[spot.index >= start_p], rates.loc[rates.index >= start_p]

### 2.2 FRED API key
FRED's keyless CSV export is often slow or blocked from cloud servers such as Colab, so we use the **official FRED API**, which needs a free key:
1. Create a free account at [fredaccount.stlouisfed.org](https://fredaccount.stlouisfed.org/apikeys) → *API Keys* → *Request API Key*.
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* → name `FRED_API_KEY`, value = your key → enable *Notebook access*.

(If you skip the secret, the cell below asks you to paste the key. Never write the key directly in the notebook: it would end up on GitHub.)

In [ ]:
try:
    from google.colab import userdata          # Colab Secrets
    FRED_API_KEY = userdata.get('FRED_API_KEY') or ''
except Exception:
    pass
if not FRED_API_KEY:
    from getpass import getpass
    FRED_API_KEY = getpass("Paste your FRED API key (leave empty to try the keyless CSV): ").strip()
print("Using FRED API" if FRED_API_KEY else "No key: using keyless CSV (may time out on Colab)")

### 2.3 Download
The printed table shows the last observation of each series: always the first check to run on real data.

In [ ]:
START = "2000-01"   # the euro exists from 1999, and momentum needs a 1-year warm-up

spot, rates = load_data(fetch=fetch_fred, start=START)
print(f"\nSample: {spot.index[0]} → {spot.index[-1]}  ({len(spot)} months)")
spot.tail()

### 2.4 First look: interest differentials vs USD
This chart is the gross "carry" each currency offers. Note how differentials move with **monetary policy cycles**: almost all collapse towards zero after 2008, then widen again with the 2022–23 Fed hikes.

In [ ]:
carry_diff = rates[spot.columns].sub(rates['USD'], axis=0)   # i* − i_USD

ax = carry_diff.plot(figsize=(12, 5), lw=1.2)
ax.axhline(0, color='k', lw=0.8)
ax.set_title("3-month interest rate differential vs USD (% p.a.)")
ax.set_ylabel("i* − i_USD (%)")
ax.legend(ncol=5, fontsize=9)
plt.show()

---
## 3. Returns and signals

### 3.1 The excess return
For each currency we compute the return of being **long foreign currency / short USD** in month $t$:
$$ xr_{t} = \ln\frac{S_{t}}{S_{t-1}} + \frac{i^*_{t-1} - i_{t-1}}{12} $$
Why this is right: in practice the trade is implemented with a **1-month forward**. By *covered interest parity*, $F_t = S_t \cdot \frac{1+i_t}{1+i^*_t}$, so buying foreign currency forward returns approximately $\ln S_{t+1} - \ln F_t \approx \Delta \ln S_{t+1} + (i^*_t - i_t)$.

### 3.2 No look-ahead bias
This is the most common backtesting mistake. The signal computed at the **end of month $t$** (with rates and prices known at $t$) is applied **only to month $t+1$ returns**. In the code this is enforced by the `shift(1)` calls.

### 3.3 Momentum
A currency's momentum at the end of month $t$ is the sum of its excess returns over the last 3 months, carry included. The signal therefore looks at the total return of the trade, not just the FX move.

In [ ]:
def compute_carry(rates: pd.DataFrame) -> pd.DataFrame:
    """Interest differential i* - i_USD (% p.a.), known at end of month t."""
    ccys = [c for c in rates.columns if c != "USD"]
    return rates[ccys].sub(rates["USD"], axis=0)


def compute_excess_returns(spot: pd.DataFrame, rates: pd.DataFrame) -> pd.DataFrame:
    """
    xr[t] = return earned DURING month t by being long foreign / short USD,
            using the interest differential known at end of month t-1.
    """
    carry = compute_carry(rates)[spot.columns]
    spot_ret = np.log(spot).diff()
    return spot_ret + carry.shift(1) / 100.0 / 12.0


def compute_momentum(xr: pd.DataFrame, lookback: int = 3) -> pd.DataFrame:
    """Cumulative excess return over the last `lookback` months (known at t)."""
    return xr.rolling(lookback, min_periods=lookback).sum()

---
## 4. Portfolio construction

1. **Carry (cross-sectional):** each month we rank the 9 currencies by interest differential. Long the 3 highest-yielders, short the 3 lowest-yielders, ±1/3 weight each. The portfolio is **dollar-neutral**: longs and shorts against USD offset, so we are effectively buying high-yielders **funded** in low-yielders (typically JPY and CHF).
2. **Carry + momentum filter:** the same portfolio, but positions with opposing momentum are set to zero.
3. **Time-series momentum (benchmark):** long if momentum is positive, short if negative, across all currencies. This tells us whether the filter adds something or whether it is "just momentum".

**Volatility targeting (optional).** Exposure is scaled to a constant ex-ante volatility (e.g. 10% p.a.) using past realised volatility. This is standard practice in systematic funds and makes strategies with different risk comparable. Leverage is capped at 3x.

**Costs:** 3 basis points per unit of turnover, a realistic-to-conservative value for institutional G10 trading. Retail costs are higher (see Limitations).

In [ ]:
def carry_weights(carry: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    """Long the n highest-yielders, short the n lowest. Weights +-1/n per leg."""
    w = pd.DataFrame(0.0, index=carry.index, columns=carry.columns)
    for t, row in carry.iterrows():
        row = row.dropna()
        if len(row) < 2 * n:
            continue
        ranked = row.sort_values()
        w.loc[t, ranked.index[-n:]] = 1.0 / n   # high yielders: LONG
        w.loc[t, ranked.index[:n]] = -1.0 / n   # low yielders (funding): SHORT
    return w


def momentum_filter(w: pd.DataFrame, mom: pd.DataFrame) -> pd.DataFrame:
    """
    Keep a carry position only if recent momentum agrees with it:
      long  & momentum < 0  -> flat
      short & momentum > 0  -> flat
    """
    mom = mom.reindex_like(w)
    keep_long = (w > 0) & (mom > 0)
    keep_short = (w < 0) & (mom < 0)
    return w.where(keep_long | keep_short, 0.0)


def ts_momentum_weights(mom: pd.DataFrame) -> pd.DataFrame:
    """Pure time-series momentum: sign of past return, equal weight."""
    sig = np.sign(mom)
    n_avail = sig.notna().sum(axis=1).replace(0, np.nan)
    return sig.div(n_avail, axis=0).fillna(0.0)


def vol_target(port_ret_gross: pd.Series, target_vol: float = 0.10,
               window: int = 24, max_lev: float = 3.0) -> pd.Series:
    """
    Leverage applied in month t+1 = target / realised vol up to month t.
    Uses only past data (shift(1)).
    """
    realised = port_ret_gross.rolling(window, min_periods=12).std() * np.sqrt(12)
    lev = (target_vol / realised).clip(upper=max_lev).shift(1)
    return lev.fillna(0.0)


def backtest(weights: pd.DataFrame, xr: pd.DataFrame, cost_bps: float = 3.0,
             leverage: pd.Series | None = None) -> pd.DataFrame:
    """
    weights at end of month t are applied to xr of month t+1.
    Transaction cost = cost_bps * turnover (sum of |dw|).
    """
    w = weights.reindex(xr.index).fillna(0.0)
    if leverage is not None:
        w = w.mul(leverage.reindex(w.index).fillna(0.0), axis=0)
    w_lag = w.shift(1).fillna(0.0)
    gross = (w_lag * xr.fillna(0.0)).sum(axis=1)
    turnover = (w - w.shift(1).fillna(0.0)).abs().sum(axis=1).shift(1).fillna(0.0)
    cost = turnover * cost_bps / 10_000
    net = gross - cost
    return pd.DataFrame({"gross": gross, "cost": cost, "net": net,
                         "turnover": turnover})

### 4.1 Performance metrics
Beyond the Sharpe ratio, the key metrics are **skewness**, **max drawdown** and **worst month**: they measure *crash risk*, the real risk of carry. Drawdowns are computed on cumulative log returns.

In [ ]:
def perf_stats(r: pd.Series) -> pd.Series:
    r = r.dropna()
    r = r[r.ne(0).cummax()]          # drop leading zeros (warm-up period)
    cum = r.cumsum()
    dd = cum - cum.cummax()
    ann_ret = r.mean() * 12
    ann_vol = r.std() * np.sqrt(12)
    return pd.Series({
        "Ann. return": ann_ret,
        "Ann. vol": ann_vol,
        "Sharpe": ann_ret / ann_vol if ann_vol > 0 else np.nan,
        "Max drawdown": dd.min(),
        "Skewness": r.skew(),
        "Worst month": r.min(),
        "Hit rate": (r > 0).mean(),
        "Months": len(r),
    })


def drawdown(r: pd.Series) -> pd.Series:
    cum = r.fillna(0).cumsum()
    return cum - cum.cummax()

In [ ]:
def run_all(spot, rates, n=3, mom_lookback=3, cost_bps=3.0, target_vol=None):
    carry = compute_carry(rates)[spot.columns]
    xr = compute_excess_returns(spot, rates)
    mom = compute_momentum(xr, mom_lookback)

    w = {
        "Carry": carry_weights(carry, n),
        "Momentum (TS)": ts_momentum_weights(mom),
    }
    w["Carry + Mom filter"] = momentum_filter(w["Carry"], mom)

    results = {}
    for name, wt in w.items():
        lev = None
        if target_vol is not None:
            gross = backtest(wt, xr, cost_bps=0.0)["gross"]
            lev = vol_target(gross, target_vol)
        results[name] = backtest(wt, xr, cost_bps=cost_bps, leverage=lev)
    return {"carry": carry, "xr": xr, "mom": mom, "weights": w, "results": results}


def current_targets(spot, rates, n=3, mom_lookback=3) -> pd.DataFrame:
    """Target weights for NEXT month, from the latest complete month."""
    out = run_all(spot, rates, n=n, mom_lookback=mom_lookback)
    t = out["weights"]["Carry + Mom filter"].index[-1]
    df = pd.DataFrame({
        "carry_%": out["carry"].loc[t],
        "momentum_%": out["mom"].loc[t] * 100,
        "carry_weight": out["weights"]["Carry"].loc[t],
        "final_weight": out["weights"]["Carry + Mom filter"].loc[t],
    })
    df.attrs["as_of"] = str(t)
    return df.sort_values("carry_%", ascending=False)

---
## 5. Results

### 5.1 Performance statistics
Two versions: **raw exposure** (±1/3 weights) and **10% volatility-targeted**, which allows a comparison at equal risk.

In [ ]:
N_LEG, LOOKBACK, COST_BPS = 3, 3, 3.0    # parameters fixed EX ANTE (standard values in the literature)

raw = run_all(spot, rates, n=N_LEG, mom_lookback=LOOKBACK, cost_bps=COST_BPS)
vt  = run_all(spot, rates, n=N_LEG, mom_lookback=LOOKBACK, cost_bps=COST_BPS, target_vol=0.10)

stats_raw = pd.DataFrame({k: perf_stats(v['net']) for k, v in raw['results'].items()})
stats_vt  = pd.DataFrame({k: perf_stats(v['net']) for k, v in vt['results'].items()})
print("=== Raw exposure (net of costs) ===")
display(stats_raw)
print("\n=== 10% volatility target (net of costs) ===")
display(stats_vt)

**How to read the table (questions to ask):**
- Does carry really show **negative skewness**? It is the "signature" of crash risk.
- Does the filter improve the **Sharpe ratio** or only the **drawdown**? These are different results. Cutting drawdowns at a similar Sharpe is already useful for a risk manager.
- Is the filter different from **pure momentum**? If pure momentum does just as well, carry adds nothing.
- How much do **costs** matter? Compare the `cost` column in the results and the turnover below.

### 5.2 Cumulative return and drawdown

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for name, res in vt['results'].items():
    r = res['net']
    r.cumsum().to_timestamp().plot(ax=axes[0], label=name, lw=1.5)
    drawdown(r).to_timestamp().plot(ax=axes[1], label=name, lw=1.2)
axes[0].set_title("Cumulative log return (10% vol target, net of costs)")
axes[0].legend()
axes[1].set_title("Drawdown")
for d, lbl in [("2008-09", "Lehman"), ("2015-01", "SNB floor"), ("2020-03", "Covid"), ("2024-08", "Yen unwind")]:
    for ax in axes:
        ax.axvline(pd.Timestamp(d), color='grey', ls='--', lw=0.8)
    axes[0].annotate(lbl, (pd.Timestamp(d), axes[0].get_ylim()[1]), fontsize=8, rotation=90, va='top', ha='right')
plt.tight_layout(); plt.show()

### 5.3 Crisis episodes: does the filter work when it matters?
This is the most important analysis. We look at the worst months in recent carry history:
- **Oct 2008**: Lehman crisis, global unwind, violent yen appreciation.
- **Aug 2011**: euro-area debt crisis, US downgrade.
- **Jan 2015**: the SNB abandons the 1.20 EUR/CHF floor; the franc jumps more than 15% in one day.
- **Mar 2020**: Covid, dash for dollars.
- **Aug 2024**: unwind of the yen-funded carry trade after the BoJ hike.

⚠️ A 3-month momentum filter is **slow** by construction. It can protect against crises that build up over several months (2008, when stress was visible from the summer), but **not** against overnight shocks (the SNB in 2015).

In [ ]:
crisis = ["2008-10", "2008-11", "2011-08", "2015-01", "2020-03", "2022-09", "2024-08"]
tab = pd.DataFrame({k: v['net'] for k, v in raw['results'].items()})
tab = tab.loc[[pd.Period(c, 'M') for c in crisis if pd.Period(c, 'M') in tab.index]] * 100
print("Monthly return in crisis months (%, raw exposure)")
display(tab.round(2))

# how much exposure did each strategy carry into the crisis month (end of previous month)?
expo = pd.DataFrame({
    'Carry gross expo': raw['weights']['Carry'].abs().sum(axis=1),
    'Filtered gross expo': raw['weights']['Carry + Mom filter'].abs().sum(axis=1)}).shift(1)
display(expo.loc[tab.index])

### 5.4 What the filter does over time
Gross exposure: 2 = all 6 carry positions active, 0 = fully flat. If the filter works as intended, it should **cut exposure before or during unwinds**.

In [ ]:
ax = expo.to_timestamp().plot(figsize=(12, 4), lw=1)
ax.set_title("Gross exposure (sum of |weights|) — 2.0 = full carry")
plt.show()

to = pd.DataFrame({k: v['turnover'] for k, v in raw['results'].items()})
print("Average monthly turnover:"); display(to.mean().to_frame('turnover'))

---
## 6. Robustness

### 6.1 Parameter sensitivity
We vary the number of currencies per leg (2, 3, 4) and the momentum lookback (1, 3, 6, 12 months).

⚠️ **Beware of overfitting.** This table checks that the result **does not hinge on one lucky parameter**. It is not a tool to pick the best cell: choosing the maximum and reporting it as the result would be *data mining*. The baseline parameters (3, 3) were fixed **before** looking at the results.

In [ ]:
grid = {}
for n in [2, 3, 4]:
    for lb in [1, 3, 6, 12]:
        res = run_all(spot, rates, n=n, mom_lookback=lb, cost_bps=COST_BPS)['results']
        grid[(n, lb)] = {
            'Carry': perf_stats(res['Carry']['net'])['Sharpe'],
            'Filtered': perf_stats(res['Carry + Mom filter']['net'])['Sharpe']}
g = pd.DataFrame(grid).T
g.index.names = ['n per leg', 'lookback (months)']
g['Δ Sharpe (filter − carry)'] = g['Filtered'] - g['Carry']
display(g.round(2))

### 6.2 Sub-samples
Carry has lived through two different "eras": before 2008 it was the king of FX strategies, then zero rates compressed differentials for over a decade. A robust result should hold in both.

In [ ]:
periods = {"2000-2007": ("2000-01", "2007-12"), "2008-2015": ("2008-01", "2015-12"),
           "2016-today": ("2016-01", str(spot.index[-1]))}
sub = {}
for lbl, (a, b) in periods.items():
    for k, v in raw['results'].items():
        sub[(lbl, k)] = perf_stats(v['net'].loc[a:b])[['Ann. return', 'Sharpe', 'Max drawdown']]
display(pd.DataFrame(sub).T.round(3))

---
## 7. Live signal: positions for next month
This table links research to implementation. It is exactly what the automation script (`src/live_signal.py`) computes every month.

- `carry_%`: interest differential vs USD
- `momentum_%`: excess return over the last 3 months
- `carry_weight`: pure carry position
- `final_weight`: position after the filter (+ = long the currency, − = short, 0 = flat)

In [ ]:
tgt = current_targets(spot, rates, n=N_LEG, mom_lookback=LOOKBACK)
print(f"Signal computed with data as of end-{tgt.attrs['as_of']} → positions for the following month\n")
display(tgt.round(3))

---
## 8. Limitations

1. **OECD interbank rates ≠ forward-implied rates.** True carry is read from *forward points*. Differences are small for G10 currencies but exist, e.g. the post-2008 *cross-currency basis*, which is a CIP violation.
2. **Monthly averages.** OECD rates are monthly averages rather than month-end values, and the latest months are forward-filled.
3. **Small universe.** With 9 currencies and 3 per leg, the portfolio is concentrated and results are noisy. Academic studies use 30–50 currencies, including emerging markets.
4. **USD excluded from the cross-section.** The dollar can never enter the long or short leg. Including it is a natural extension.
5. **Costs and retail implementation.** A retail broker does not pay the theoretical carry: on FX spot positions it applies its own benchmark rate ± a spread, so net carry is lower, especially for small accounts.
6. **A sample with very different regimes** (zero rates 2009–2021). The past may not repeat.
7. **No formal statistical test.** A Sharpe ratio of 0.5 over 25 years has a standard error of about 0.2. A natural extension is a bootstrap of Sharpe differences.

## 9. Possible extensions
- Add a **value** signal (deviation from PPP) → a "carry + momentum + value" portfolio à la Asness, Moskowitz & Pedersen (2013).
- Condition carry on **global volatility** (VIX or implied FX volatility): switch carry off above a threshold.
- Condition on the **Fed policy cycle**: does carry work better during US easing or tightening phases? A direct link to monetary policy spillovers.
- Use actual **forward points** (Bloomberg/Refinitiv access via CBS).

---
### References
- Fama, E. (1984). Forward and spot exchange rates. *Journal of Monetary Economics*.
- Brunnermeier, M., Nagel, S. & Pedersen, L. (2008). Carry trades and currency crashes. *NBER Macroeconomics Annual*.
- Lustig, H., Roussanov, N. & Verdelhan, A. (2011). Common risk factors in currency markets. *Review of Financial Studies*.
- Menkhoff, L., Sarno, L., Schmeling, M. & Schrimpf, A. (2012). Carry trades and global foreign exchange volatility. *Journal of Finance*.
- Menkhoff, L., Sarno, L., Schmeling, M. & Schrimpf, A. (2012). Currency momentum strategies. *Journal of Financial Economics*.
- Asness, C., Moskowitz, T. & Pedersen, L. (2013). Value and momentum everywhere. *Journal of Finance*.